In [9]:
import os
import sys
import slideio
import numpy as np
import cv2

# ==========================================
# 1. USER CONFIGURATION
# ==========================================
IMAGE_PATH = "/Users/nadavyayon/Downloads/20260424_wga_test1_wellview_c2_afterstain_011B2.vsi"

CYTOPLASM_CHANNEL_NAME = "GFP"  
NUCLEUS_CHANNEL_NAME = "DAPI"
EXPECTED_DIAMETER_MICRONS = 50.0  
NUC_DIAMETER_MICRONS = 15.0

# ==========================================
# 2. FILE OPENING & SCENE SELECTION
# ==========================================
print(f"[PREVIEW] Opening VSI bundle via slideio: {IMAGE_PATH}")

try:
    slide = slideio.open_slide(IMAGE_PATH, "VSI")
except Exception as e:
    print(f"-> FATAL ERROR: Could not open VSI file. ({e})")
    sys.exit(1)

target_scene = None
max_pixels = 0

print("Scanning internal VSI scenes...")
for i in range(slide.num_scenes):
    scene = slide.get_scene(i)
    
    width, height = scene.size
    pixels = width * height
    print(f" -> Found Scene [{i}]: Dimensions = {width}x{height} ({scene.num_channels} channels) | Name: {scene.name}")
    
    if pixels > max_pixels:
        max_pixels = pixels
        target_scene = scene

if target_scene is None:
    print("-> FATAL ERROR: No valid image scenes found.")
    sys.exit(1)

img_width, img_height = target_scene.size
total_bands = target_scene.num_channels
print(f"\n[SUCCESS] Locked onto Scene: '{target_scene.name}' ({img_width}x{img_height})")

# ==========================================
# 3. METADATA EXTRACTION 
# ==========================================
channel_names = [target_scene.get_channel_name(i).upper() for i in range(total_bands)]

cellpose_cyto_idx = 0
cellpose_nucl_idx = 0
for i, name in enumerate(channel_names):
    if CYTOPLASM_CHANNEL_NAME.upper() in name:
        cellpose_cyto_idx = i + 1 
    if NUCLEUS_CHANNEL_NAME.upper() in name:
        cellpose_nucl_idx = i + 1

CHANNELS = [cellpose_cyto_idx, cellpose_nucl_idx]

resolution_tuple = target_scene.resolution 
if resolution_tuple is not None and resolution_tuple[0] > 0:
    PIXEL_SIZE_MICRONS = resolution_tuple[0] * 1_000_000 
else:
    PIXEL_SIZE_MICRONS = 0.25
DIAMETER_PX = EXPECTED_DIAMETER_MICRONS / PIXEL_SIZE_MICRONS
NUC_DIAMETER_PX = NUC_DIAMETER_MICRONS / PIXEL_SIZE_MICRONS

print("\n" + "="*50)
print("      AUTOMATED METADATA VERIFICATION REPORT")
print("="*50)
print(f"Slide Dimensions     : {img_width}x{img_height}")
print(f"Detected Channels    : {', '.join(channel_names)}")
print(f"Selected Cyto        : '{CYTOPLASM_CHANNEL_NAME}' -> Index {cellpose_cyto_idx}")
print(f"Selected Nucleus     : '{NUCLEUS_CHANNEL_NAME}' -> Index {cellpose_nucl_idx}")
print(f"Cellpose CHANNELS    : {CHANNELS}")
print(f"Extracted Scale      : {PIXEL_SIZE_MICRONS:.4f} µm/pixel")
print(f"Model Diameter       : {DIAMETER_PX:.2f} pixels")
print("="*50 + "\n")

# ==========================================
# 4. LOW-RES THUMBNAIL VISUALIZATION
# ==========================================
print("Generating low-res thumbnail from master resolution...")
thumb_width = 1000
thumb_height = int(img_height * (thumb_width / img_width))

thumb_np = target_scene.read_block((0, 0, img_width, img_height), size=(thumb_width, thumb_height))

# FIX: slideio already natively outputs standard (Height, Width, Channels) shape.
# We only need to check if it's a 2D array and expand it if missing the channel dimension.
if len(thumb_np.shape) == 2:
    thumb_np = np.expand_dims(thumb_np, axis=2)

if thumb_np.dtype == np.uint16:
    thumb_np = (thumb_np / 256).astype(np.uint8)

preview_img = np.zeros((thumb_height, thumb_width, 3), dtype=np.uint8)
if cellpose_cyto_idx > 0 and cellpose_cyto_idx <= thumb_np.shape[2]:
    preview_img[:, :, 2] = thumb_np[:, :, cellpose_cyto_idx - 1] # Red
if cellpose_nucl_idx > 0 and cellpose_nucl_idx <= thumb_np.shape[2]:
    preview_img[:, :, 0] = thumb_np[:, :, cellpose_nucl_idx - 1] # Blue

cv2.imwrite("./slide_preview.png", preview_img)
print("-> SUCCESS: Thumbnail saved to './slide_preview.png'. Everything is configured.")

[PREVIEW] Opening VSI bundle via slideio: /Users/nadavyayon/Downloads/20260424_wga_test1_wellview_c2_afterstain_011B2.vsi
Scanning internal VSI scenes...
 -> Found Scene [0]: Dimensions = 35597x35597 (3 channels) | Name: DAPI, Cy5, GFP

[SUCCESS] Locked onto Scene: 'DAPI, Cy5, GFP' (35597x35597)

      AUTOMATED METADATA VERIFICATION REPORT
Slide Dimensions     : 35597x35597
Detected Channels    : DAPI, CY5, GFP
Selected Cyto        : 'GFP' -> Index 3
Selected Nucleus     : 'DAPI' -> Index 1
Cellpose CHANNELS    : [3, 1]
Extracted Scale      : 0.1621 µm/pixel
Model Diameter       : 308.53 pixels

Generating low-res thumbnail from master resolution...
-> SUCCESS: Thumbnail saved to './slide_preview.png'. Everything is configured.


In [ ]:
# ==========================================
# 1. ENVIRONMENT & MODEL CONFIGURATION
# ==========================================
import os
import sys
import json
import cv2
import numpy as np
from cellpose import models
from shapely.geometry import Polygon, box

# Force system configurations to prevent OpenMP multi-threading collisions
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

# --- DYNAMIC NAMING BASED ON YOUR IMAGE FILE ---
# Update this with your actual image path or extraction logic
# (e.g., IMAGE_FILE_PATH = target_scene.get_path() or similar)
IMAGE_FILE_PATH = "/Users/nadavyayon/Downloads/20260424_wga_test1_wellview_c2_afterstain_011B2.vsi"

# Extract the base image name without directory paths or extensions
image_base_name = os.path.splitext(os.path.basename(IMAGE_FILE_PATH))[0]
output_dir = os.path.dirname(IMAGE_FILE_PATH)

# Dynamically construct output paths using the exact image name
OUTPUT_CELLS_GEOJSON = os.path.join(output_dir, f"{image_base_name}_cells.geojson")
OUTPUT_NUC_GEOJSON   = os.path.join(output_dir, f"{image_base_name}_nuclei.geojson")

# Sliding window tile parameters
TILE_SIZE = 2048   
MARGIN = 256       
STRIDE = TILE_SIZE - (2 * MARGIN)  

print("[PIPELINE] Initializing Cellpose 3 Cytoplasm Model (cyto3)...")
model_cells = models.CellposeModel(gpu=True, model_type='cyto3')

print("[PIPELINE] Initializing Cellpose Nuclei Model (nuclei)...")
model_nuc = models.CellposeModel(gpu=True, model_type='nuclei')

all_cell_features = []
all_nuc_features = []
cell_counter = 0

# ==========================================
# 2. THE SLIDING-WINDOW TILED PROCESSING LOOP
# ==========================================
print(f"[PIPELINE] Starting segmentation loop across Region of Interest...")

PERCENTAGE = 0.3
TARGET_WIDTH = int(img_width * PERCENTAGE)
TARGET_HEIGHT = int(img_height * PERCENTAGE)

TARGET_START_X = (img_width - TARGET_WIDTH) // 2
TARGET_START_Y = (img_height - TARGET_HEIGHT) // 2
TARGET_END_X = TARGET_START_X + TARGET_WIDTH
TARGET_END_Y = TARGET_START_Y + TARGET_HEIGHT

# Expand the loop bounds outward by MARGIN (256px) to protect edges
ROI_START_X = max(0, TARGET_START_X - MARGIN)
ROI_START_Y = max(0, TARGET_START_Y - MARGIN)
ROI_END_X = min(img_width, TARGET_END_X + MARGIN)
ROI_END_Y = min(img_height, TARGET_END_Y + MARGIN)

# --- CONFIGURATION SANITY CHECK & TUNING ---
# try:
# if 'DIAMETER_PX' not in globals() or DIAMETER_PX < 1.0:
    # DIAMETER_PX = 30.0 / 0.1621  # Default cell size (~185 pixels)


print(f"-> Target Center Window       : X=[{TARGET_START_X} to {TARGET_END_X}], Y=[{TARGET_START_Y} to {TARGET_END_Y}]")
print(f"-> Cell Inference Settings    | CHANNELS: {CHANNELS} | DIAMETER: {DIAMETER_PX:.1f} px")
print(f"-> Nucleus Inference Settings | Pure Grayscale Slicing | DIAMETER: {NUC_DIAMETER_PX:.1f} px")
# except Exception as e:
# print(f"-> State Error: Make sure Part 1 executed successfully right before this cell! ({e})")
# sys.exit(1)


# Sliding window loops
for y in range(ROI_START_Y, ROI_END_Y, STRIDE):
    for x in range(ROI_START_X, ROI_END_X, STRIDE):
        
        w = min(TILE_SIZE, ROI_END_X - x)
        h = min(TILE_SIZE, ROI_END_Y - y)
        
        if w < (2 * MARGIN) or h < (2 * MARGIN):
            continue
            
        img_np = target_scene.read_block((x, y, w, h))
        
        if len(img_np.shape) == 2:
            img_np = np.expand_dims(img_np, axis=2)
            
        if img_np.dtype == np.uint16:
            img_np = (img_np / 256).astype(np.uint8)
        
        if img_np.shape[0] == 0 or img_np.shape[1] == 0:
            continue

        print(f" -> Segmenting Tile at X: {x}, Y: {y}")
        
        # 1. Segment whole cells using the full multi-channel tile
        masks_cells, _, _ = model_cells.eval(img_np, channels=CHANNELS, diameter=DIAMETER_PX)
        
        # 2. Slice out ONLY the DAPI channel to prevent cross-channel leakage
        if CHANNELS[1] > 0:
            nuc_channel_idx = CHANNELS[1] - 1  
        else:
            nuc_channel_idx = 0  
            
        img_nuc = img_np[:, :, nuc_channel_idx]
        
        # Evaluate Nuclei Boundaries using the isolated grayscale DAPI image
        masks_nuc, _, _ = model_nuc.eval(img_nuc, channels=[0, 0], diameter=NUC_DIAMETER_PX)
        
        local_valid_box = box(MARGIN, MARGIN, w - MARGIN, h - MARGIN)
        unique_cell_ids = np.unique(masks_cells)
        unique_cell_ids = unique_cell_ids[unique_cell_ids != 0] 
        
        for obj_id in unique_cell_ids:
            # 1. PROCESS THE CELL MEMBRANE
            cell_mask = (masks_cells == obj_id).astype(np.uint8) * 255
            contours, _ = cv2.findContours(cell_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            
            if not contours:
                continue
                
            cell_contour = max(contours, key=cv2.contourArea)
            if len(cell_contour) < 3:
                continue  
            
            local_poly = Polygon(cell_contour.reshape(-1, 2).tolist()).buffer(0)
            if local_poly.geom_type == 'MultiPolygon':
                local_poly = max(local_poly.geoms, key=lambda a: a.area)
            
            # Filter out edges using window margin logic
            if local_poly.area < 15 or not local_valid_box.contains(local_poly.centroid):
                continue 
            
            local_poly = local_poly.simplify(0.2, preserve_topology=True)
            if local_poly.is_empty or not isinstance(local_poly, Polygon):
                continue
            
            global_cell_points = [[pt[0] + x, pt[1] + y] for pt in list(local_poly.exterior.coords)]
            global_cell_points.append(global_cell_points[0]) 
            
            # Generate cell ID string based on base image name to keep them contextualized
            cell_counter += 1
            shared_object_id = f"{image_base_name}_cell_{cell_counter}"
            
            # 2. MATCH THE INTERNAL NUCLEUS
            global_nuc_points = None
            nuc_ids_in_cell = masks_nuc[masks_cells == obj_id]
            nuc_ids_in_cell = nuc_ids_in_cell[nuc_ids_in_cell != 0]
            
            if len(nuc_ids_in_cell) > 0:
                best_nuc_id = np.bincount(nuc_ids_in_cell).argmax()
                nuc_mask = (masks_nuc == best_nuc_id).astype(np.uint8) * 255
                nuc_contours, _ = cv2.findContours(nuc_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                
                if nuc_contours:
                    nuc_contour = max(nuc_contours, key=cv2.contourArea)
                    if len(nuc_contour) >= 3:
                        nuc_poly = Polygon(nuc_contour.reshape(-1, 2).tolist()).buffer(0)
                        if nuc_poly.geom_type == 'MultiPolygon':
                            nuc_poly = max(nuc_poly.geoms, key=lambda a: a.area)
                            
                        nuc_poly = nuc_poly.simplify(0.2, preserve_topology=True)
                        if not nuc_poly.is_empty and isinstance(nuc_poly, Polygon):
                            global_nuc_points = [[pt[0] + x, pt[1] + y] for pt in list(nuc_poly.exterior.coords)]
                            global_nuc_points.append(global_nuc_points[0])

            # 3. APPEND CELL OBJECT
            cell_feature = {
                "type": "Feature",
                "id": shared_object_id,
                "geometry": {
                    "type": "Polygon",
                    "coordinates": [global_cell_points]
                },
                "properties": {
                    "objectType": "detection",
                    "name": shared_object_id,
                    "classification": {
                        "name": "Cell",
                        "color": [59, 150, 72]
                    }
                }
            }
            all_cell_features.append(cell_feature)
            
            # 4. APPEND NUCLEUS OBJECT WITH LINKED ID
            if global_nuc_points is not None:
                nuc_feature = {
                    "type": "Feature",
                    "id": f"{image_base_name}_nucleus_{cell_counter}", 
                    "geometry": {
                        "type": "Polygon",
                        "coordinates": [global_nuc_points]
                    },
                    "properties": {
                        "objectType": "detection",
                        "name": f"{image_base_name}_nucleus_{cell_counter}",
                        "Parent_Cell_Id": shared_object_id,  
                        "classification": {
                            "name": "Nucleus",
                            "color": [0, 0, 255]
                        }
                    }
                }
                all_nuc_features.append(nuc_feature)

# ==========================================
# 3. COMPILING & DISK EXPORT (TWO FILES)
# ==========================================
print(f"\n[PIPELINE] Complete! Found {cell_counter} linked pairs.")

# Export Cells
cells_geojson_data = {"type": "FeatureCollection", "features": all_cell_features}
with open(OUTPUT_CELLS_GEOJSON, 'w') as f:
    json.dump(cells_geojson_data, f)
print(f"-> Saved Cells GeoJSON to: {OUTPUT_CELLS_GEOJSON}")

# Export Nuclei
nuc_geojson_data = {"type": "FeatureCollection", "features": all_nuc_features}
with open(OUTPUT_NUC_GEOJSON, 'w') as f:
    json.dump(nuc_geojson_data, f)
print(f"-> Saved Nuclei GeoJSON to: {OUTPUT_NUC_GEOJSON}")

[PIPELINE] Initializing Cellpose 3 Cytoplasm Model (cyto3)...
[PIPELINE] Initializing Cellpose Nuclei Model (nuclei)...
[PIPELINE] Starting segmentation loop across Region of Interest...
-> Target Center Window       : X=[14239 to 21358], Y=[14239 to 21358]
-> Cell Inference Settings    | CHANNELS: [3, 1] | DIAMETER: 308.5 px
-> Nucleus Inference Settings | Pure Grayscale Slicing | DIAMETER: 92.6 px
 -> Segmenting Tile at X: 13983, Y: 13983
 -> Segmenting Tile at X: 15519, Y: 13983
 -> Segmenting Tile at X: 17055, Y: 13983
 -> Segmenting Tile at X: 18591, Y: 13983
 -> Segmenting Tile at X: 20127, Y: 13983
 -> Segmenting Tile at X: 13983, Y: 15519
 -> Segmenting Tile at X: 15519, Y: 15519
 -> Segmenting Tile at X: 17055, Y: 15519
 -> Segmenting Tile at X: 18591, Y: 15519
 -> Segmenting Tile at X: 20127, Y: 15519
 -> Segmenting Tile at X: 13983, Y: 17055
 -> Segmenting Tile at X: 15519, Y: 17055
 -> Segmenting Tile at X: 17055, Y: 17055
 -> Segmenting Tile at X: 18591, Y: 17055
 -> Segme